**UPLOADING THE DATASET**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving History (1).json to History (1).json


**DATA PROCESSING**

In [ ]:
import json
from datetime import datetime

input_file = "History (1).json"
output_file = "titles_with_timestamp.txt"

def convert_time_usec_to_readable(time_usec):
    time_seconds = time_usec / 1000000
    dt_object = datetime.fromtimestamp(time_seconds)
    return dt_object.strftime("%Y-%m-%d %H:%M:%S")

# Load JSON
with open(input_file, "r") as file:
    data = json.load(file)

# Extract title + timestamp
entries = [
    (entry["title"], entry["time_usec"])
    for entry in data.get("Browser History", [])
    if entry["title"].strip() != ""
]

# ✅ Remove duplicates (important)
unique_entries = list(set(entries))

# Write to file
with open(output_file, "w") as file:
    for title, time_usec in unique_entries:
        readable_timestamp = convert_time_usec_to_readable(time_usec)
        file.write(f"{readable_timestamp}: {title}\n")

print(f"Extracted {len(unique_entries)} cleaned entries")

Extracted 231 cleaned entries


**FORMATTING THE DATA**

In [ ]:
# Read cleaned data
with open("titles_with_timestamp.txt", "r") as file:
    lines = file.readlines()

# 🔥 Define broader suspicious keywords (expanded list)
keywords = [
    "hack", "hacker", "attack", "exploit", "malware", "virus",
    "crypto", "bitcoin", "torrent", "piracy", "crack",
    "dark web", "leak", "breach", "password", "phishing",
    "anonymous", "ddos", "spoof"
]

# 🔹 Extract suspicious entries
suspicious = [line for line in lines if any(k in line.lower() for k in keywords)]

# 🔹 Fallback if no suspicious entries found
if len(suspicious) == 0:
    print("⚠️ No suspicious activity found. Using general browsing data instead.\n")
    selected = lines[:1000]
else:
    print(f"✅ Suspicious entries found: {len(suspicious)}\n")
    selected = suspicious[:1000]  # limit if too many

# 🔹 Format for LLM
formatted_data = "User Browser Activity Analysis:\n\n"

for line in selected:
    formatted_data += f"- {line.strip()}\n"

# Preview
print(formatted_data[:1000])

⚠️ No suspicious activity found. Using general browsing data instead.

User Browser Activity Analysis:

- 2024-03-13 15:21:45: ChatGPT as a Copilot for Investigating Digital Evidence
- 2024-03-13 15:41:56: accounts.google.com/signin/oauth/consent?as=S788831419:1710344511541446&authuser=0&client_id=1062961139910-l2m55cb9h51u5cuc9c56eb3fevouidh9.apps.googleusercontent.com&flowName=GeneralOAuthFlow&part=AJi8hAODwliWu_VDeCVitGw-3-_-RmZAb3LwKbt7OH133iGY-psy3puerz4YQQ1DUf-HqbOEvyYpdqs-_-kAOwL-uX-ScSN5FnWy2-nLOBN3z2NLvTSWHz8m3M6963CyM1h2I8vk1qOtFrAHBu61z-O0SqYfkrBYMYk11hXcSjEtiPa2LQGsglpIn5Iaz_3a2c6n-A3Gt-o9F7-0Hv48eHzYjzlmNLNqgW7TIHnHuah1tLPzmjyiRZOOj54o3_UyQgeQopocmONZaw8Uk2S2a7vuS2V8m4jM1S6vY4WRZ95DFAcTAxinCQTwBei7ckiZ_cXrtGAAxOmFixewagLafrXLw1JsiON_K3khiSfDtv2m_x-dwhMCpNA2yqv3ROe-u-NSjtnfOoP8vhKf99GjzdxEg1kY90O3xwTGnFCfnvNrj_ZQw4Sa5pX1tAxuGAMaQJVsh3axxDHpF8LLVIneC7gTmzDgzTxLtlako99HO6xeTBB7yCBf2LDnIIWxKojBLHyj1yhIb_LIeSFm5GOzJMaQi3Slmf4F9Xf7_nHyFt757qiERtHPcFuXVRqetRHENIoPtNLGPMPcaF9b_06h

In [ ]:
with open("titles_with_timestamp.txt", "r") as file:
    final_lines = file.readlines()

**TOP INTERESTS**

In [ ]:
categories = {
    "youtube": "entertainment",
    "netflix": "entertainment",
    "amazon": "shopping",
    "flipkart": "shopping",
    "linkedin": "professional",
    "github": "technology",
    "stack overflow": "technology",
    "reddit": "social",
    "instagram": "social"
}

from collections import Counter

category_count = Counter()

for line in final_lines:
    line_lower = line.lower()
    for key in categories:
        if key in line_lower:
            category_count[categories[key]] += 1

interests = [cat for cat, _ in category_count.most_common(3)]

print("Top Interests:", interests)

Top Interests: ['entertainment', 'shopping', 'social']


**SUSPICIOUS ACTIVITY COUNT**

In [ ]:
suspicious_keywords = [
    "hack", "hacker", "attack", "exploit", "malware",
    "crypto", "bitcoin", "torrent", "crack",
    "dark web", "leak", "breach", "phishing", "ddos"
]

suspicious_count = 0

for line in final_lines:
    for word in suspicious_keywords:
        if word in line.lower():
            suspicious_count += 1

print("Suspicious activity count:", suspicious_count)

Suspicious activity count: 0


**ADDING SUSPICIOUS LINES**




In [ ]:
test_suspicious = [
    "2024-03-10 02:30:00: how to hack wifi password",
    "2024-03-11 01:15:00: dark web access tutorial",
    "2024-03-12 03:45:00: torrent cracked software download"
]

final_lines.extend(test_suspicious)

**UPDATED COUNT**

In [ ]:
suspicious_count = 0

for line in final_lines:
    for word in suspicious_keywords:
        if word in line.lower():
            suspicious_count += 1

print("Suspicious activity count:", suspicious_count)

Suspicious activity count: 4


**RISK LEVEL**

In [ ]:
if suspicious_count >= 5:
    risk = "High"
elif suspicious_count >= 2:
    risk = "Medium"
else:
    risk = "Low"

print("Risk Level:", risk)

Risk Level: Medium


**NIGHT ACTIVITY**


In [ ]:
night_activity = 0

for line in final_lines:
    time_part = line.split(":")[0]   # get date + hour
    hour = int(time_part.split(" ")[1].split(":")[0])

    if 0 <= hour <= 4:
        night_activity += 1

if night_activity > 5:
    behavior = "Night-active"
else:
    behavior = "Normal"

print("Behavior:", behavior)

Behavior: Night-active


**INSTALLING TRANSFORMER**

In [ ]:
!pip install -q transformers accelerate

**USING MODEL - TINY LLAMA**

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

**PROMPT**

In [ ]:
prompt = f"""
<|system|>
You are a cybersecurity analyst and behavioral profiler.

Generate a realistic and professional psychological profile based only on given data.

Rules:
- Do NOT assign any name to the individual
- Do NOT make accusations (e.g., criminal, illegal activity)
- Keep analysis neutral and evidence-based
- Avoid exaggeration

<|user|>
Data:
Interests: {interests}
Behavior: {behavior}
Risk Level: {risk}
Suspicious Activity Count: {suspicious_count}

Generate:
- Psychological profile (detailed paragraph)
- Age range
-Location
- Interests explanation
- Behavioral insights
- Cybersecurity risk reasoning

<|assistant|>
"""

**RESULT**

In [ ]:
result = pipe(prompt, max_new_tokens=500, do_sample=True)[0]["generated_text"]

print(result.split("<|assistant|>")[-1])

Both `max_new_tokens` (=500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Interests:
- Enjoys playing video games, watching movies, and exploring new places
- Is a fan of fashion and often attends events and shows
- Loves to shop online for new clothing and accessories
- Prefers socializing with friends over social media

Behavior:
- A night-active individual, preferring to go to the gym or the movies after work
- Strictly avoids social media and other online platforms during the day
- Has a propensity for shopping online, particularly for new clothing and accessories
- Often attends events, such as concerts or fashion shows, instead of socializing online
- Has a tendency to spend a lot of money on clothes and shoes

Risk Level:
- Medium risk, as they may be susceptible to phishing or other types of online fraud
- Has previously been victims of online scams or phishing attempts
- Is familiar with common online security vulnerabilities and takes steps to protect themselves

Suspicious Activity Count:
- Has been asked to join a group chat for a specific group

**EVALUATION REPORT**

In [ ]:
true_age = "20-29"
true_interests = ["entertainment", "education", "social"]

In [ ]:
def age_score(true, pred):
    if true == pred:
        return 1
    elif abs(int(true.split('-')[0]) - int(pred.split('-')[0])) == 10:
        return 0.5
    else:
        return 0

predicted_age = "20-29"  # from your output

print("Age Score:", age_score(true_age, predicted_age))

Age Score: 1


In [ ]:
predicted_interests = interests

In [ ]:
correct = len(set(true_interests) & set(predicted_interests))
total = max(len(true_interests), len(predicted_interests))

interest_score = correct / total

print("Interest Score:", interest_score)

Interest Score: 0.6666666666666666


In [ ]:
print("\n Evaluation Results:\n")

print(f"{'Metric':<12}{'Truth':<40}{'Prediction':<40}{'Score'}")

# Age
age_s = age_score(true_age, predicted_age)
print(f"{'Age':<12}{true_age:<40}{predicted_age:<40}{age_s}")

# Interests
true_int_str = ", ".join(true_interests)
pred_int_str = ", ".join(predicted_interests)
print(f"{'Interests':<12}{true_int_str:<40}{pred_int_str:<40}{round(interest_score,2)}")

# Location (you can define this)
true_location = "Maryland"
predicted_location = "California"

location_score = 0  # since not matching

print(f"{'Location':<12}{true_location:<40}{predicted_location:<40}{location_score}")


 Evaluation Results:

Metric      Truth                                   Prediction                              Score
Age         20-29                                   20-29                                   1
Interests   entertainment, education, social        entertainment, shopping, social         0.67
Location    Maryland                                California                              0
